# Bài tập thực hành 2: Xây dựng cây quyết định và rừng cây trên dữ liệu bệnh tiểu đường

**Mục tiêu**
- Khám phá và tiền xử lý bộ dữ liệu *Diabetes Prediction Dataset* (100.000 bản ghi, 9 cột).
- Xây dựng mô hình **Cây quyết định (Decision Tree)** và **Rừng cây ngẫu nhiên (Random Forest)** để dự đoán bệnh tiểu đường (`diabetes` = 0/1).
- Điều chỉnh siêu tham số, đánh giá và so sánh các mô hình.

**Mô tả các cột**

| Cột | Ý nghĩa |
|---|---|
| gender | Giới tính |
| age | Tuổi |
| hypertension | Có tăng huyết áp (0/1) |
| heart_disease | Có bệnh tim (0/1) |
| smoking_history | Tiền sử hút thuốc |
| bmi | Chỉ số khối cơ thể |
| HbA1c_level | Mức HbA1c (đường huyết trung bình 2-3 tháng) |
| blood_glucose_level | Mức đường huyết hiện tại |
| diabetes | **Nhãn cần dự đoán** (0: không, 1: có) |

## 1. Import thư viện

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 2. Đọc dữ liệu

In [ ]:
# Nếu chạy trên Google Colab: tải file lên bằng đoạn lệnh dưới (bỏ dấu # để dùng)
# from google.colab import files
# files.upload()

df = pd.read_csv("diabetes_prediction_dataset.csv")
print("Kích thước dữ liệu:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

## 3. Khám phá dữ liệu (EDA)

In [ ]:
print("Giá trị thiếu mỗi cột:")
print(df.isnull().sum())
print("\nSố dòng trùng lặp:", df.duplicated().sum())
print("\nCác giá trị của gender:\n", df["gender"].value_counts())
print("\nCác giá trị của smoking_history:\n", df["smoking_history"].value_counts())

In [ ]:
# Phân bố nhãn
counts = df["diabetes"].value_counts()
print(counts)
print("Tỉ lệ mắc bệnh: {:.2%}".format(counts[1] / len(df)))

plt.figure(figsize=(4, 3.5))
sns.countplot(x="diabetes", data=df)
plt.title("Phân bố nhãn diabetes")
plt.show()

In [ ]:
num_cols = ["age", "bmi", "HbA1c_level", "blood_glucose_level"]

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, col in zip(axes.ravel(), num_cols):
    sns.histplot(data=df, x=col, hue="diabetes", bins=40, kde=False, ax=ax, element="step")
    ax.set_title("Phân bố " + col + " theo diabetes")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, col in zip(axes, num_cols):
    sns.boxplot(data=df, x="diabetes", y=col, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
cat_cols = ["gender", "smoking_history", "hypertension", "heart_disease"]
fig, axes = plt.subplots(1, 4, figsize=(17, 4))
for ax, col in zip(axes, cat_cols):
    rate = df.groupby(col)["diabetes"].mean().sort_values(ascending=False)
    sns.barplot(x=rate.index.astype(str), y=rate.values, ax=ax)
    ax.set_title("Tỉ lệ mắc bệnh theo " + col)
    ax.set_ylabel("Tỉ lệ diabetes")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(df.select_dtypes("number").corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Ma trận tương quan")
plt.show()

## 4. Tiền xử lý dữ liệu

- Loại bỏ các dòng trùng lặp.
- Mã hóa biến phân loại `gender`, `smoking_history` bằng one-hot encoding (cây quyết định của scikit-learn chỉ nhận dữ liệu số).
- Chia tập train/test theo tỉ lệ 80/20, dùng `stratify` để giữ nguyên tỉ lệ nhãn vì dữ liệu mất cân bằng.

> Cây quyết định và rừng cây **không cần chuẩn hóa/scale** dữ liệu số do chúng chỉ so sánh ngưỡng.

In [ ]:
data = df.drop_duplicates().reset_index(drop=True)
print("Sau khi loại trùng:", data.shape)

X = pd.get_dummies(data.drop(columns="diabetes"), columns=["gender", "smoking_history"], drop_first=False)
y = data["diabetes"]
print("Số đặc trưng sau mã hóa:", X.shape[1])
print(list(X.columns))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Tỉ lệ dương (train): {:.2%} | (test): {:.2%}".format(y_train.mean(), y_test.mean()))

### Hàm hỗ trợ đánh giá mô hình

Vì dữ liệu mất cân bằng (khoảng 8-9% mắc bệnh) nên **không chỉ dựa vào accuracy**; ta theo dõi thêm Precision, Recall, F1 và ROC-AUC của lớp bệnh (1).

In [ ]:
results = {}   # lưu kết quả để so sánh ở cuối

def evaluate(name, model, X_tr=None, X_te=None, show=True):
    X_tr = X_train if X_tr is None else X_tr
    X_te = X_test if X_te is None else X_te
    pred_tr = model.predict(X_tr)
    pred = model.predict(X_te)
    proba = model.predict_proba(X_te)[:, 1]
    res = {
        "Train Acc": accuracy_score(y_train, pred_tr),
        "Test Acc": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, proba),
    }
    results[name] = res
    if show:
        print("=== " + name + " ===")
        for k, v in res.items():
            print("{:<10}: {:.4f}".format(k, v))
        print()
        print(classification_report(y_test, pred, target_names=["Không bệnh", "Có bệnh"]))
        ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=["Không", "Có"]).plot(cmap="Blues")
        plt.title("Ma trận nhầm lẫn - " + name)
        plt.show()
    return res

## 5. Cây quyết định (Decision Tree)

### 5.1 Mô hình cơ sở (không giới hạn độ sâu)
Cây không giới hạn sẽ phát triển đến khi các lá thuần nhất, dễ dẫn đến **overfitting** (accuracy train rất cao nhưng test thấp hơn).

In [ ]:
dt_base = DecisionTreeClassifier(random_state=RANDOM_STATE)
dt_base.fit(X_train, y_train)
print("Độ sâu cây:", dt_base.get_depth(), "| Số lá:", dt_base.get_n_leaves())
evaluate("Decision Tree (mặc định)", dt_base)

### 5.2 So sánh tiêu chí phân tách: Gini và Entropy

In [ ]:
for crit in ["gini", "entropy"]:
    m = DecisionTreeClassifier(criterion=crit, max_depth=5, random_state=RANDOM_STATE).fit(X_train, y_train)
    evaluate("DT depth=5 (" + crit + ")", m, show=False)

pd.DataFrame(results).T.loc[["DT depth=5 (gini)", "DT depth=5 (entropy)"]].round(4)

### 5.3 Ảnh hưởng của độ sâu `max_depth`
Vẽ accuracy và F1 trên train/test theo độ sâu để tìm điểm cân bằng giữa underfitting và overfitting.

In [ ]:
depths = range(1, 26)
rows = []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE).fit(X_train, y_train)
    rows.append({
        "depth": d,
        "train_acc": accuracy_score(y_train, m.predict(X_train)),
        "test_acc": accuracy_score(y_test, m.predict(X_test)),
        "train_f1": f1_score(y_train, m.predict(X_train)),
        "test_f1": f1_score(y_test, m.predict(X_test)),
    })
depth_df = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(depth_df["depth"], depth_df["train_acc"], "o-", label="Train")
axes[0].plot(depth_df["depth"], depth_df["test_acc"], "s-", label="Test")
axes[0].set_title("Accuracy theo max_depth"); axes[0].set_xlabel("max_depth"); axes[0].legend()
axes[1].plot(depth_df["depth"], depth_df["train_f1"], "o-", label="Train")
axes[1].plot(depth_df["depth"], depth_df["test_f1"], "s-", label="Test")
axes[1].set_title("F1 (lớp bệnh) theo max_depth"); axes[1].set_xlabel("max_depth"); axes[1].legend()
plt.tight_layout()
plt.show()

best_row = depth_df.loc[depth_df["test_f1"].idxmax()]
print("Độ sâu cho F1 test cao nhất: {} (F1 = {:.4f})".format(int(best_row["depth"]), best_row["test_f1"]))

### 5.4 Tinh chỉnh siêu tham số bằng GridSearchCV
Tìm tổ hợp tốt nhất của `max_depth`, `min_samples_split`, `min_samples_leaf`, `criterion` bằng kiểm định chéo 5 phần (stratified), tối ưu theo **F1**.

In [ ]:
param_grid_dt = {
    "criterion": ["gini", "entropy"],
    "max_depth": [3, 4, 5, 6, 8, 10],
    "min_samples_split": [2, 20, 100],
    "min_samples_leaf": [1, 10, 50],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid_dt = GridSearchCV(DecisionTreeClassifier(random_state=RANDOM_STATE),
                       param_grid_dt, scoring="f1", cv=cv, n_jobs=-1)
grid_dt.fit(X_train, y_train)

print("Tham số tốt nhất:", grid_dt.best_params_)
print("F1 CV tốt nhất: {:.4f}".format(grid_dt.best_score_))

In [ ]:
dt_best = grid_dt.best_estimator_
evaluate("Decision Tree (tối ưu)", dt_best)

### 5.5 Trực quan hóa cây quyết định

In [ ]:
plt.figure(figsize=(22, 9))
plot_tree(dt_best, max_depth=3, feature_names=X.columns, class_names=["Không bệnh", "Có bệnh"],
          filled=True, rounded=True, fontsize=9)
plt.title("Cây quyết định tối ưu (hiển thị 3 mức đầu)")
plt.show()

In [ ]:
# Luật quyết định dạng văn bản (3 mức đầu)
print(export_text(dt_best, feature_names=list(X.columns), max_depth=3))

## 6. Rừng cây ngẫu nhiên (Random Forest)

Random Forest huấn luyện nhiều cây trên các mẫu bootstrap và chọn ngẫu nhiên tập đặc trưng ở mỗi lần tách, sau đó biểu quyết kết quả, giúp **giảm phương sai/overfitting** so với một cây đơn lẻ.

### 6.1 Mô hình cơ sở

In [ ]:
rf_base = RandomForestClassifier(n_estimators=100, oob_score=True, n_jobs=-1, random_state=RANDOM_STATE)
rf_base.fit(X_train, y_train)
print("OOB score: {:.4f}".format(rf_base.oob_score_))
evaluate("Random Forest (mặc định)", rf_base)

### 6.2 Ảnh hưởng của số cây `n_estimators`

In [ ]:
n_list = [1, 5, 10, 25, 50, 100, 200, 300]
rows = []
for n in n_list:
    m = RandomForestClassifier(n_estimators=n, n_jobs=-1, random_state=RANDOM_STATE).fit(X_train, y_train)
    rows.append({"n_estimators": n,
                 "test_acc": accuracy_score(y_test, m.predict(X_test)),
                 "test_f1": f1_score(y_test, m.predict(X_test)),
                 "test_auc": roc_auc_score(y_test, m.predict_proba(X_test)[:, 1])})
n_df = pd.DataFrame(rows)
display(n_df.round(4))

plt.figure(figsize=(7, 4))
plt.plot(n_df["n_estimators"], n_df["test_f1"], "o-", label="F1")
plt.plot(n_df["n_estimators"], n_df["test_auc"], "s-", label="ROC-AUC")
plt.xlabel("n_estimators"); plt.title("Hiệu năng theo số cây"); plt.legend()
plt.show()

### 6.3 Tinh chỉnh siêu tham số bằng RandomizedSearchCV
Dùng tìm kiếm ngẫu nhiên để tiết kiệm thời gian (không gian tham số của Random Forest lớn).

In [ ]:
param_dist_rf = {
    "n_estimators": [100, 200, 300],
    "max_depth": [5, 8, 10, 15, None],
    "min_samples_split": [2, 10, 50],
    "min_samples_leaf": [1, 5, 20],
    "max_features": ["sqrt", "log2", 0.5],
}
search_rf = RandomizedSearchCV(RandomForestClassifier(n_jobs=-1, random_state=RANDOM_STATE),
                               param_dist_rf, n_iter=12, scoring="f1", cv=3,
                               random_state=RANDOM_STATE, n_jobs=1)
search_rf.fit(X_train, y_train)

print("Tham số tốt nhất:", search_rf.best_params_)
print("F1 CV tốt nhất: {:.4f}".format(search_rf.best_score_))

In [ ]:
rf_best = search_rf.best_estimator_
evaluate("Random Forest (tối ưu)", rf_best)

## 7. Độ quan trọng của đặc trưng (Feature Importance)

In [ ]:
imp = pd.DataFrame({
    "Decision Tree": pd.Series(dt_best.feature_importances_, index=X.columns),
    "Random Forest": pd.Series(rf_best.feature_importances_, index=X.columns),
}).sort_values("Random Forest", ascending=False)

top = imp.head(10)
top.plot(kind="barh", figsize=(9, 5)).invert_yaxis()
plt.title("Top 10 đặc trưng quan trọng nhất")
plt.xlabel("Độ quan trọng")
plt.tight_layout()
plt.show()
imp.head(10).round(4)

## 8. Xử lý mất cân bằng lớp bằng `class_weight="balanced"`
Lớp bệnh chỉ chiếm khoảng 8-9% nên mô hình có xu hướng ưu tiên lớp đa số. Thử gán trọng số cân bằng để tăng **Recall** (bắt được nhiều ca bệnh hơn).

In [ ]:
dt_bal = DecisionTreeClassifier(**grid_dt.best_params_, class_weight="balanced", random_state=RANDOM_STATE).fit(X_train, y_train)
evaluate("Decision Tree (balanced)", dt_bal, show=False)

rf_params = {k: v for k, v in search_rf.best_params_.items()}
rf_bal = RandomForestClassifier(**rf_params, class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE).fit(X_train, y_train)
evaluate("Random Forest (balanced)", rf_bal, show=False)

pd.DataFrame(results).T.loc[["Decision Tree (tối ưu)", "Decision Tree (balanced)",
                             "Random Forest (tối ưu)", "Random Forest (balanced)"]].round(4)

## 9. So sánh các mô hình

In [ ]:
summary = pd.DataFrame(results).T.round(4)
summary.sort_values("F1", ascending=False)

In [ ]:
plt.figure(figsize=(7, 6))
models = {
    "Decision Tree (tối ưu)": dt_best,
    "Random Forest (tối ưu)": rf_best,
    "Decision Tree (balanced)": dt_bal,
    "Random Forest (balanced)": rf_bal,
}
for name, m in models.items():
    fpr, tpr, _ = roc_curve(y_test, m.predict_proba(X_test)[:, 1])
    plt.plot(fpr, tpr, label="{} (AUC = {:.3f})".format(name, roc_auc_score(y_test, m.predict_proba(X_test)[:, 1])))
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("Đường cong ROC"); plt.legend(loc="lower right")
plt.show()

In [ ]:
metrics = ["Test Acc", "Precision", "Recall", "F1", "ROC-AUC"]
summary.loc[["Decision Tree (tối ưu)", "Random Forest (tối ưu)",
             "Decision Tree (balanced)", "Random Forest (balanced)"], metrics].plot(
    kind="bar", figsize=(11, 5), rot=15)
plt.title("So sánh các chỉ số giữa các mô hình")
plt.ylim(0, 1.05)
plt.legend(loc="lower right", ncol=3)
plt.show()

## 10. Kết luận

- **Dữ liệu**: mất cân bằng (~8,8% mắc bệnh), vì vậy cần xem Precision/Recall/F1/ROC-AUC chứ không chỉ accuracy.
- **Cây quyết định**: cây không giới hạn độ sâu bị overfitting (accuracy train ≈ 1 nhưng F1 trên test thấp hơn). Giới hạn `max_depth` và tăng `min_samples_leaf` giúp mô hình tổng quát hóa tốt hơn và dễ diễn giải bằng các luật if-then.
- **Rừng cây ngẫu nhiên**: thường cho ROC-AUC và độ ổn định cao hơn một cây đơn lẻ nhờ kết hợp nhiều cây; tăng số cây cải thiện hiệu năng rồi bão hòa.
- **Đặc trưng quan trọng nhất**: `HbA1c_level` và `blood_glucose_level`, tiếp theo là `age` và `bmi`, phù hợp với kiến thức y khoa về chẩn đoán tiểu đường.
- **Mất cân bằng lớp**: `class_weight="balanced"` làm tăng Recall (bắt được nhiều ca bệnh hơn) nhưng giảm Precision; tùy bài toán (sàng lọc y tế thường ưu tiên Recall) mà chọn cấu hình phù hợp.

> Hãy đối chiếu các nhận xét trên với bảng kết quả ở mục 9 khi chạy trên máy của bạn để điền số liệu cụ thể vào báo cáo.